# ST-SCKM Example: California-like Wildfire Data

This notebook demonstrates the reference ST-SCKM workflow using the bundled synthetic sample data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src import STSCKM
from src.clustering import assign_risk_labels, evaluate_labels
from src.utils import add_default_features, standardize_features

In [ ]:
df = pd.read_csv("data/sample_data.csv")
df = add_default_features(df)

X_spatial, _ = standardize_features(df, ["x_proj", "y_proj"])
X_temporal, _ = standardize_features(df, ["time_days"])

model = STSCKM(n_clusters=4, spatial_weight=0.5, temporal_weight=1.5, lambda_spatial=1.0, n_neighbors=5, random_state=42)
df["cluster"] = model.fit_predict(X_spatial, X_temporal)
df["risk_zone"] = assign_risk_labels(df, "cluster", "log_frp")
df[["latitude", "longitude", "datetime", "frp", "cluster", "risk_zone"]].head()

In [ ]:
X_eval = pd.concat([pd.DataFrame(X_spatial), pd.DataFrame(X_temporal)], axis=1).to_numpy()
evaluate_labels(X_eval, df["cluster"].to_numpy())

In [ ]:
palette = {"Low Risk": "#1f77b4", "Moderate Risk": "#2ca02c", "High Risk": "#ff7f0e", "Extreme Risk": "#d62728"}
fig, ax = plt.subplots(figsize=(7, 8))
for risk, group in df.groupby("risk_zone"):
    ax.scatter(group["longitude"], group["latitude"], s=14, alpha=0.7, color=palette.get(risk, "gray"), label=risk)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("ST-SCKM Sample Risk Zones")
ax.legend(frameon=True)
plt.show()